# Geolife GPS Visualization and Discretization

This notebook visualizes Geolife GPS trajectories, fixed-grid discretization, and codebook-style discretization for representative users and transportation modes.

It focuses on four common modes: `walk`, `bike`, `bus`, and `subway`.

Outputs in this notebook:
- Raw GPS plots for representative users.
- Fixed `1.5 km` grid discretization plots.
- Codebook-style discretization plots with predictability annotations.
- A mode-level comparison of predictability.

If `geopandas` is available, raw GPS points are rendered with `GeoDataFrame` objects. Otherwise the notebook falls back to ordinary `matplotlib` plotting.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    from shapely.geometry import Point
    HAS_GEOPANDAS = True
except ImportError:
    gpd = None
    Point = None
    HAS_GEOPANDAS = False

ROOT = Path.cwd()
if ROOT.name != "Quantifying-the-Predictability-of-Travel-Sequences":
    ROOT = ROOT / "Quantifying-the-Predictability-of-Travel-Sequences"

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from geolife_preprocess import (
    build_resampled_dataset,
    extract_segment_arrays,
    grid_discretize,
    symbolic_ctw_metrics,
)

DATA_DIR = ROOT.parent / "Geolife Trajectories 1.3" / "Data"
SELECTED_USERS = ["128", "153", "163"]
SELECTED_MODES = ["walk", "bike", "bus", "subway"]
SAMPLE_INTERVAL_MIN = 15
INTERPOLATION_METHOD = "linear"
GRID_SIZE_KM = 1.5
CODEBOOK_SIZE = 12
FIGSIZE = (6, 6)
RANDOM_SEED = 7

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)

print(f"Notebook root: {ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"geopandas available: {HAS_GEOPANDAS}")

In [ ]:
def reindex_symbols(sequence):
    mapping = {}
    out = []
    next_id = 0
    for token in sequence:
        token = int(token)
        if token not in mapping:
            mapping[token] = next_id
            next_id += 1
        out.append(mapping[token])
    return np.asarray(out, dtype=int)


def fit_simple_kmeans(xy_km, num_codes=12, max_iter=30, seed=7):
    xy_km = np.asarray(xy_km, dtype=float)
    if len(xy_km) == 0:
        return np.array([], dtype=int), np.empty((0, 2), dtype=float)
    n_codes = min(num_codes, len(xy_km))
    rng = np.random.default_rng(seed)
    init_idx = rng.choice(len(xy_km), size=n_codes, replace=False)
    centroids = xy_km[init_idx].copy()

    for _ in range(max_iter):
        distances = ((xy_km[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        labels = distances.argmin(axis=1)
        new_centroids = centroids.copy()
        for code_id in range(n_codes):
            mask = labels == code_id
            if mask.any():
                new_centroids[code_id] = xy_km[mask].mean(axis=0)
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids

    distances = ((xy_km[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    labels = distances.argmin(axis=1)
    return labels.astype(int), centroids


def codebook_discretize(xy_km, num_codes=12, seed=7):
    labels, centroids = fit_simple_kmeans(xy_km, num_codes=num_codes, seed=seed)
    if len(labels) == 0:
        return {
            "tokens": np.array([], dtype=int),
            "centroids_km": np.empty((0, 2), dtype=float),
            "assigned_centroids_km": np.empty((0, 2), dtype=float),
            "distortion_km": np.array([], dtype=float),
        }
    assigned = centroids[labels]
    distortion_km = np.sqrt(((xy_km - assigned) ** 2).sum(axis=1))
    return {
        "tokens": labels,
        "centroids_km": centroids,
        "assigned_centroids_km": assigned,
        "distortion_km": distortion_km,
    }


def build_geodataframe(df):
    if not HAS_GEOPANDAS:
        return None
    geometry = [Point(lon, lat) for lon, lat in zip(df["longitude"], df["latitude"])]
    return gpd.GeoDataFrame(df.copy(), geometry=geometry, crs="EPSG:4326")


def choose_representative_segments(samples_df, selected_modes, selected_users=None):
    candidates = samples_df.copy()
    if selected_users is not None:
        candidates = candidates[candidates["user_id"].isin(selected_users)]

    grouped = (
        candidates.groupby(["segment_id", "user_id", "trajectory_id", "mode"], as_index=False)
        .agg(
            num_points=("timestamp", "size"),
            start_time=("timestamp", "min"),
            end_time=("timestamp", "max"),
        )
        .sort_values(["mode", "num_points", "user_id"], ascending=[True, False, True])
    )

    chosen_rows = []
    used_users = set()
    for mode in selected_modes:
        subset = grouped[grouped["mode"] == mode]
        if subset.empty:
            continue
        preferred = subset[~subset["user_id"].isin(used_users)]
        row = preferred.iloc[0] if not preferred.empty else subset.iloc[0]
        chosen_rows.append(row)
        used_users.add(row["user_id"])

    chosen = pd.DataFrame(chosen_rows)
    if chosen.empty:
        return chosen
    return chosen.sort_values(["user_id", "mode"]).reset_index(drop=True)


def compute_mode_predictability(chosen_segments, segment_arrays):
    segment_lookup = {segment["summary"].segment_id: segment for segment in segment_arrays}
    rows = []
    for row in chosen_segments.itertuples(index=False):
        segment = segment_lookup[row.segment_id]
        xy_km = np.asarray(segment["xy_km"], dtype=float)
        direct = grid_discretize(xy_km, GRID_SIZE_KM)
        direct_metrics = symbolic_ctw_metrics(reindex_symbols(direct["tokens"]))
        codebook = codebook_discretize(xy_km, num_codes=CODEBOOK_SIZE, seed=RANDOM_SEED)
        codebook_metrics = symbolic_ctw_metrics(reindex_symbols(codebook["tokens"]))
        rows.append(
            {
                "segment_id": row.segment_id,
                "user_id": row.user_id,
                "trajectory_id": row.trajectory_id,
                "mode": row.mode,
                "num_points": row.num_points,
                "direct_predictability": direct_metrics["predictability"],
                "direct_entropy_bits": direct_metrics["entropy_rate_bits"],
                "codebook_predictability": codebook_metrics["predictability"],
                "codebook_entropy_bits": codebook_metrics["entropy_rate_bits"],
            }
        )
    return pd.DataFrame(rows)


def plot_raw_gps(ax, df, title):
    if HAS_GEOPANDAS:
        gdf = build_geodataframe(df)
        gdf.plot(ax=ax, column="mode", legend=False, markersize=12, alpha=0.85)
    else:
        for mode, group in df.groupby("mode"):
            ax.scatter(group["longitude"], group["latitude"], s=12, label=mode, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


def plot_grid_discretization(ax, df, xy_km, title):
    direct = grid_discretize(xy_km, GRID_SIZE_KM)
    centroids = direct["centroids_km"]
    ax.scatter(df["longitude"], df["latitude"], s=10, c=direct["tokens"], cmap="tab20", alpha=0.75)
    lon_min, lon_max = df["longitude"].min(), df["longitude"].max()
    lat_min, lat_max = df["latitude"].min(), df["latitude"].max()
    ax.set_xlim(lon_min - 0.01, lon_max + 0.01)
    ax.set_ylim(lat_min - 0.01, lat_max + 0.01)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    x_min = xy_km[:, 0].min()
    x_max = xy_km[:, 0].max()
    y_min = xy_km[:, 1].min()
    y_max = xy_km[:, 1].max()
    gx = np.arange(np.floor(x_min / GRID_SIZE_KM) * GRID_SIZE_KM, np.ceil(x_max / GRID_SIZE_KM) * GRID_SIZE_KM + GRID_SIZE_KM, GRID_SIZE_KM)
    gy = np.arange(np.floor(y_min / GRID_SIZE_KM) * GRID_SIZE_KM, np.ceil(y_max / GRID_SIZE_KM) * GRID_SIZE_KM + GRID_SIZE_KM, GRID_SIZE_KM)
    if len(gx) > 1 and len(gy) > 1:
        lon_scale = (lon_max - lon_min) / max(x_max - x_min, 1e-9)
        lat_scale = (lat_max - lat_min) / max(y_max - y_min, 1e-9)
        for x0 in gx[:-1]:
            for y0 in gy[:-1]:
                rect_lon = lon_min + (x0 - x_min) * lon_scale
                rect_lat = lat_min + (y0 - y_min) * lat_scale
                width = GRID_SIZE_KM * lon_scale
                height = GRID_SIZE_KM * lat_scale
                ax.add_patch(Rectangle((rect_lon, rect_lat), width, height, fill=False, lw=0.4, ec="gray", alpha=0.35))
    return direct


def plot_codebook_discretization(ax, df, xy_km, title):
    codebook = codebook_discretize(xy_km, num_codes=CODEBOOK_SIZE, seed=RANDOM_SEED)
    ax.scatter(df["longitude"], df["latitude"], s=10, c=codebook["tokens"], cmap="tab10", alpha=0.65)
    if len(codebook["assigned_centroids_km"]) > 0:
        assigned = codebook["assigned_centroids_km"]
        x_min, x_max = xy_km[:, 0].min(), xy_km[:, 0].max()
        y_min, y_max = xy_km[:, 1].min(), xy_km[:, 1].max()
        lon_min, lon_max = df["longitude"].min(), df["longitude"].max()
        lat_min, lat_max = df["latitude"].min(), df["latitude"].max()
        lon_assigned = lon_min + (assigned[:, 0] - x_min) * ((lon_max - lon_min) / max(x_max - x_min, 1e-9))
        lat_assigned = lat_min + (assigned[:, 1] - y_min) * ((lat_max - lat_min) / max(y_max - y_min, 1e-9))
        ax.scatter(lon_assigned, lat_assigned, s=24, c=codebook["tokens"], cmap="tab10", marker="x", alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    return codebook

In [ ]:
samples_df = build_resampled_dataset(
    data_dir=DATA_DIR,
    sample_intervals=[SAMPLE_INTERVAL_MIN],
    interpolation_methods=[INTERPOLATION_METHOD],
    require_mode=True,
    min_points=8,
    selected_user_ids=SELECTED_USERS,
)

samples_df = samples_df[
    samples_df["mode"].isin(SELECTED_MODES)
].sort_values(["user_id", "segment_id", "timestamp"]).copy()

block_start = (
    samples_df["segment_id"].ne(samples_df["segment_id"].shift()) |
    samples_df["mode"].ne(samples_df["mode"].shift())
)
samples_df["mode_block_num"] = block_start.cumsum().astype(int)
samples_df["mode_block_id"] = samples_df["segment_id"] + "_block" + samples_df["mode_block_num"].astype(str)

mode_blocks = (
    samples_df.groupby(["mode_block_id", "segment_id", "user_id", "trajectory_id", "mode"], as_index=False)
    .agg(
        num_points=("timestamp", "size"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
    )
    .sort_values(["mode", "num_points", "user_id"], ascending=[True, False, True])
)

chosen_rows = []
for mode in SELECTED_MODES:
    subset = mode_blocks[mode_blocks["mode"] == mode]
    if subset.empty:
        continue
    row = subset.iloc[0]
    chosen_rows.append(row)

chosen_segments = pd.DataFrame(chosen_rows).reset_index(drop=True)

predictability_rows = []
for row in chosen_segments.itertuples(index=False):
    block_df = samples_df[samples_df["mode_block_id"] == row.mode_block_id].sort_values("timestamp")
    xy_km = block_df[["x_km", "y_km"]].to_numpy(dtype=float)
    direct = grid_discretize(xy_km, GRID_SIZE_KM)
    codebook = codebook_discretize(xy_km, num_codes=CODEBOOK_SIZE, seed=RANDOM_SEED)
    direct_metrics = symbolic_ctw_metrics(reindex_symbols(direct["tokens"]))
    codebook_metrics = symbolic_ctw_metrics(reindex_symbols(codebook["tokens"]))
    predictability_rows.append(
        {
            "mode_block_id": row.mode_block_id,
            "direct_predictability": direct_metrics["predictability"],
            "direct_entropy_bits": direct_metrics["entropy_rate_bits"],
            "direct_alphabet_size": direct_metrics["alphabet_size"],
            "codebook_predictability": codebook_metrics["predictability"],
            "codebook_entropy_bits": codebook_metrics["entropy_rate_bits"],
        }
    )

predictability_df = pd.DataFrame(predictability_rows)
chosen_segments = chosen_segments.merge(predictability_df, on="mode_block_id", how="left")

print(f"Samples: {len(samples_df):,}")
print(f"Mode blocks: {samples_df['mode_block_id'].nunique():,}")
display(chosen_segments)

dominant_mode_summary = (
    chosen_segments[["user_id", "mode", "direct_predictability", "codebook_predictability", "num_points"]]
    .sort_values(["mode", "user_id"])
)
display(dominant_mode_summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for ax, row in zip(axes, chosen_segments.itertuples(index=False)):
    block_df = samples_df[samples_df["mode_block_id"] == row.mode_block_id].sort_values("timestamp")
    title = (
        f"User {row.user_id} | {row.mode}\n"
        f"Fixed-grid predictability={row.direct_predictability:.3f}"
    )
    plot_raw_gps(ax, block_df, title)
    if not HAS_GEOPANDAS:
        ax.legend(loc="best", fontsize=8)

for ax in axes[len(chosen_segments):]:
    ax.axis("off")

fig.suptitle("Raw Geolife GPS Mode Blocks", fontsize=16)
fig.tight_layout()
plt.show()

mode_plot = chosen_segments.sort_values("mode")
fig, ax = plt.subplots(figsize=(8, 4))
bar_x = np.arange(len(mode_plot))
width = 0.35
ax.bar(bar_x - width / 2, mode_plot["direct_predictability"], width=width, label="Fixed grid")
ax.bar(bar_x + width / 2, mode_plot["codebook_predictability"], width=width, label="Codebook style")
ax.set_xticks(bar_x)
ax.set_xticklabels([f"{row.mode}\nU{row.user_id}" for row in mode_plot.itertuples(index=False)])
ax.set_ylabel("Predictability")
ax.set_title("Predictability by transportation mode")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(chosen_segments), 2, figsize=(12, 5 * max(len(chosen_segments), 1)))
if len(chosen_segments) == 1:
    axes = np.asarray([axes])

for row_idx, row in enumerate(chosen_segments.itertuples(index=False)):
    block_df = samples_df[samples_df["mode_block_id"] == row.mode_block_id].sort_values("timestamp")
    xy_km = block_df[["x_km", "y_km"]].to_numpy(dtype=float)
    direct_preview = grid_discretize(xy_km, GRID_SIZE_KM)

    plot_grid_discretization(
        axes[row_idx, 0],
        block_df,
        xy_km,
        title=(
            f"User {row.user_id} | {row.mode} | fixed grid\n"
            f"pi={row.direct_predictability:.3f}, alphabet={int(direct_preview['alphabet_size'])}"
        ),
    )

    plot_codebook_discretization(
        axes[row_idx, 1],
        block_df,
        xy_km,
        title=(
            f"User {row.user_id} | {row.mode} | codebook style\n"
            f"pi={row.codebook_predictability:.3f}, codes={CODEBOOK_SIZE}"
        ),
    )

fig.tight_layout()
plt.show()

In [ ]:
latest_runs = sorted((ROOT / "geolife_results").glob("geolife_*"))
latest_run = latest_runs[-1] if latest_runs else None
vq_path = latest_run / "latent_code_occurrences.csv" if latest_run else None

if latest_run is None or not vq_path.exists():
    print("No previous Geolife run directory was found. The notebook is using codebook-style quantization only.")
else:
    vq_df = pd.read_csv(vq_path)
    if vq_df.empty or vq_df["raw_code"].dropna().empty:
        print(f"Found {latest_run.name}, but there are no VQ latent assignments yet. Codebook-style plots remain active.")
    else:
        display(vq_df.head())
        print(
            "Real VQ latent assignments are available in latent_code_occurrences.csv. "
            "You can replace the codebook-style visualization with those assignments after running the full VQ experiment."
        )

## Notes

- `Fixed grid` uses the same `1.5 km` spatial tolerance discussed in the Geolife external validation plan.
- `Codebook style` here means a small centroid codebook learned directly from the selected segment for visualization. This is useful even before the full VQ-VAE run is available.
- Once the full VQ-VAE experiment is executed with PyTorch installed, the notebook can switch to the saved latent assignments in `geolife_results/*/latent_code_occurrences.csv` for a true model-based codebook visualization.